# PyRIT Agentic Red-Teaming POC — Notebook

Runs the same catalog + engine as the web app (`scripts/serve.py`).  
Three examples — all start in **demo mode** (no network); Example B shows the live-mode swap.

**Container (recommended):**
```
docker run --rm -it \
  -e PYTHONPATH=/work \
  -v <repo>:/work -w /work \
  -p 8888:8888 \
  ghcr.io/vamshikadumuri/pyrit:0.13.0-v2 \
  jupyter notebook --ip=0.0.0.0 --no-browser
```

**Locally:** activate `pyritpocvenv`, run `pip install notebook`, launch from the repo root.

---

In [ ]:
import sys, uuid, asyncio
from pathlib import Path

# Add project root to sys.path for local runs.
# In the container PYTHONPATH=/work handles this; the insert is a safe no-op.
_root = Path().resolve()
if _root.name == "notebooks":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

print(f"Python {sys.version.split()[0]}  root={_root}")

## 1. Load the Catalog

157 plugins · 35 strategies · 10 framework presets, ingested from `promptfoo_plugins_catalog_1.xlsx`.  
This is the same catalog the web app loads at startup.

In [ ]:
from agentic_redteam.catalog.loader import load_catalog

catalog = load_catalog()
print(f"plugins={len(catalog.plugins)}  strategies={len(catalog.strategies)}  presets={len(catalog.presets)}")
print("\nPresets:", list(catalog.presets))
print("\nPlugin groups:")
for group, plugins in sorted(catalog.plugins_by_group().items()):
    print(f"  {group:30s} {len(plugins):3d}")

---

## Example A — Run the OWASP Agentic Preset (Demo Mode)

Uses the deterministic `demo_executor_factory` — no network, no LLM gateway required.  
Swap to `real_executor_factory` (Example B) for live runs in the container.

The `AppProfile` describes the target application; it grounds objective generation  
and feeds the rubric grader's `purpose` / `tools` / `entities` bindings.

In [ ]:
from agentic_redteam.config import ModelConfig
from agentic_redteam.engine.plan import RunConfig
from agentic_redteam.engine.profile import AppProfile
from agentic_redteam.records import RunRequest
from agentic_redteam.store import Store
from agentic_redteam.orchestrator import Orchestrator
from agentic_redteam.web.demo import demo_executor_factory, demo_llm_factory

profile = AppProfile(
    purpose="A banking chatbot that executes transactions and answers account queries.",
    tools=["transfer_funds", "get_balance", "get_statement"],
    roles=["customer", "relationship_manager"],
    sensitive_data_types=["account_number", "transaction_history"],
)

preset   = catalog.presets["owasp_agentic"]
run_id   = f"nb-a-{uuid.uuid4().hex[:6]}"
config   = RunConfig(
    run_id=run_id,
    plugin_ids=preset.plugins[:5],           # use preset.plugins for a full 22-plugin run
    strategy_ids=preset.recommended_strategies[:1],
    profile=profile,
    n=3,                                     # objectives generated per plugin
)
fake     = ModelConfig(endpoint="http://demo/v1", model_name="demo")
request  = RunRequest(config=config, target=fake, judge=fake, requested_by="notebook")

store        = Store(":memory:")             # pass a file path (e.g. "runs.db") to persist
orchestrator = Orchestrator(
    catalog, store,
    llm=demo_llm_factory(request),
    executor=demo_executor_factory(request),
)
print("plugins :", config.plugin_ids)
print("strategy:", config.strategy_ids[0])

In [ ]:
# Jupyter supports top-level `await` (IPykernel >= 5.0 / Python 3.11).
# In a .py script wrap in: summary = asyncio.run(orchestrator.run(request))
summary = await orchestrator.run(request)
print(f"status={summary.status}  total={summary.total}  succeeded={summary.succeeded}  ASR={summary.asr:.0%}")

In [ ]:
from agentic_redteam.reports.aggregation import build_report

records = store.get_executions(run_id)
report  = build_report(records)

print(f"Overall ASR : {report['overall_asr']:.1%}")
print(f"Executions  : {report['total_executions']}  ({report['errors']} errors)")
print(f"Findings    : {len(report['findings'])}")
print()

sc = report["framework_scorecard"].get("owasp_agentic", {})
if sc:
    print("OWASP Agentic scorecard:")
    for code, cell in sorted(sc.items()):
        bar = "\u2588" * int(cell["asr"] * 10)
        print(f"  {code:14s} {bar:<10s} {cell['asr']:.0%}  ({cell['succeeded']}/{cell['total']})")

---

## Example B — Custom Crescendo (Live Mode)

Refactors `crescendo.py` through the engine.  
**Requires the `ghcr.io/vamshikadumuri/pyrit:0.13.0-v2` container** and live gateway/attacker endpoints.

Set these environment variables before running:

| Variable | Description | Default |
|---|---|---|
| `TARGET_ENDPOINT` | Internal OpenAI-compatible gateway | `https://stork.sp.uat.dbs.corp/v1` |
| `TARGET_MODEL` | Target model name / ID | `69a17167fb3315370dbf866a` |
| `OPENAI_CHAT_KEY` | Gateway API key | — |
| `ATTACKER_ENDPOINT` | Local vLLM endpoint | `http://host.docker.internal:8001/v1` |
| `ATTACKER_MODEL` | Attacker model name | `Qwen3.6-35B-A3B-4bit` |
| `DEMO_MODE` | `1` to skip real calls (demo executor) | `0` |

In demo mode (`DEMO_MODE=1`) the cell runs fully offline.

In [ ]:
import os

TARGET_ENDPOINT   = os.environ.get("TARGET_ENDPOINT",   "https://stork.sp.uat.dbs.corp/v1")
TARGET_MODEL      = os.environ.get("TARGET_MODEL",      "69a17167fb3315370dbf866a")
ATTACKER_ENDPOINT = os.environ.get("ATTACKER_ENDPOINT", "http://host.docker.internal:8001/v1")
ATTACKER_MODEL    = os.environ.get("ATTACKER_MODEL",    "Qwen3.6-35B-A3B-4bit")
DEMO_MODE         = os.environ.get("DEMO_MODE", "0") == "1"

if DEMO_MODE:
    print("DEMO_MODE=1 — demo executor (no network). Unset DEMO_MODE for a live run.")
else:
    print(f"Live mode: target={TARGET_MODEL!r}  attacker={ATTACKER_MODEL!r}")

In [ ]:
run_id_b = f"nb-b-{uuid.uuid4().hex[:6]}"
config_b = RunConfig(
    run_id=run_id_b,
    plugin_ids=["agentic:indirect-prompt-injection"],
    strategy_ids=["crescendo"],
    profile=AppProfile(
        purpose="A banking chatbot that executes transactions and answers account queries.",
        tools=["transfer_funds", "get_balance"],
    ),
    n=1,
)
target_b  = ModelConfig(endpoint=TARGET_ENDPOINT, model_name=TARGET_MODEL,
                         api_key_env="OPENAI_CHAT_KEY")
attack_b  = ModelConfig(endpoint=ATTACKER_ENDPOINT, model_name=ATTACKER_MODEL)
request_b = RunRequest(
    config=config_b, target=target_b, judge=target_b, adversarial=attack_b,
    requested_by="notebook-crescendo",
)

store_b = Store(":memory:")
if DEMO_MODE:
    executor_b = demo_executor_factory(request_b)
    llm_b      = demo_llm_factory(request_b)
else:
    from agentic_redteam.web.live import real_executor_factory, real_llm_factory
    executor_b = real_executor_factory(request_b)
    llm_b      = real_llm_factory(request_b)

orch_b    = Orchestrator(catalog, store_b, llm=llm_b, executor=executor_b)
summary_b = await orch_b.run(request_b)
print(f"status={summary_b.status}  succeeded={summary_b.succeeded}/{summary_b.total}")

In [ ]:
for rec in store_b.get_executions(run_id_b):
    print(f"plugin={rec.plugin_id}  strategy={rec.strategy_id}  status={rec.status}  fidelity={rec.fidelity}")
    if rec.rationale:
        print(f"  rationale: {rec.rationale[:120]}")
    if rec.conversation_id:
        print(f"  conversation_id: {rec.conversation_id}")

---

## Example C — Explore Results

Query the store and build a full report.  
Uses the records from Example A; you can substitute `run_id_b` or any persisted run.  
Pass a file path to `Store("runs.db")` in Example A to make runs persist across sessions.

In [ ]:
print("Runs in store A:")
for row in store.list_runs():
    print(f"  {row['run_id']}  status={row['status']}")

print("\nRuns in store B:")
for row in store_b.list_runs():
    print(f"  {row['run_id']}  status={row['status']}")

In [ ]:
all_records = store.get_executions(run_id)
full_report = build_report(all_records)

print("=== Framework Scorecard ===")
for family, codes in full_report["framework_scorecard"].items():
    if codes:
        print(f"\n{family.upper()}")
        for code, stats in sorted(codes.items()):
            bar = "\u2588" * int(stats["asr"] * 10)
            print(f"  {code:14s} {bar:<10s} {stats['asr']:.0%}  ({stats['succeeded']}/{stats['total']})")

print("\n=== Findings (first 5) ===")
for f in full_report["findings"][:5]:
    print(f"  [{f['severity']}] {f['plugin_id']}  via {f['strategy_id']}")
    print(f"    {f['objective'][:80]}")

print("\n=== Sanity Flags ===")
for flag in full_report["sanity_flags"]:
    print(f"  {flag['plugin_id']}  ASR={flag['asr']:.0%}  ({flag['note']})")
if not full_report["sanity_flags"]:
    print("  (none)")